# Mehfooz: AI-Powered GLOF Early Warning Pipeline Exploration

This notebook demonstrates the end-to-end telemetry and hazard detection pipeline for **Glacial Lake Outburst Floods (GLOFs)** in Northern Pakistan.

### Pipeline Steps:
1. **Sentinel-2 Satellite Image Acquisition / Synthetic Generation**
2. **Water Masking & Lake Surface Area Measurement**
3. **Vision-Language Analysis (Qwen-VL Heuristic)**
4. **Weighted Multi-Factor Risk Assessment Engine**
5. **Multilingual Alert Translation (Urdu, Sindhi, Pashto) & Voice Synthesis**

In [ ]:
import os
import sys
import numpy as np
from PIL import Image, ImageDraw

# Ensure app module is in path
sys.path.insert(0, os.path.abspath('..'))

from app.regions import REGIONS, get_region
from app.risk_engine import calculate_risk
from app.translator import translate_alert
from app.tts import synthesize_voice
from app.pipeline import run_pipeline

print("Available Monitored Glacial Lakes in Pakistan:")
for rid, info in REGIONS.items():
    print(f" - {info['name']} (Lat: {info['lat']}, Lon: {info['lon']})")

## 1. Satellite Imagery Ingestion & Water Mask Heuristic
In full mode, Sentinel Hub fetches 10m-resolution Sentinel-2 L2A optical bands (RGB & NIR). In demo mode, a synthetic lake model with randomized geomorphic expansion is rendered.

In [ ]:
from app.satellite_ingest import fetch_satellite_image
from app.qwen_analyzer import analyze_image

# Fetch satellite snapshot for Shishper Glacial Lake
img_path = fetch_satellite_image("shishper_lake", "2026-08-28")
print(f"Satellite Image generated/saved at: {img_path}")

# Run pixel-level water segmentation and scene inspection
analysis = analyze_image(img_path)
print("Satellite Analysis Results:")
for k, v in analysis.items():
    print(f"  {k}: {v}")

## 2. Multi-Factor Risk Scoring Engine
The risk scoring model evaluates:
- **Surface Area Expansion Factor** (relative to historical baseline)
- **New Outflow Channels / Moraine Breaches**
- **Snowmelt Acceleration**

Formula: `Risk = 0.5 * AreaScore * Expansion + 0.3 * ChannelScore + 0.2 * MeltScore`

In [ ]:
# Compare baseline vs surging lake scenarios
baseline_analysis = {
    "lake_area_km2": 8.5,
    "new_channels": False,
    "snowmelt_acceleration": "low"
}

surging_analysis = {
    "lake_area_km2": 14.8,
    "new_channels": True,
    "snowmelt_acceleration": "high"
}

risk_baseline = calculate_risk(baseline_analysis, last_area_km2=8.5)
risk_surging = calculate_risk(surging_analysis, last_area_km2=8.5)

print("Baseline Scenario Risk:", risk_baseline)
print("Surging Glacial Lake Scenario Risk:", risk_surging)

## 3. Multilingual Alert Translation (Urdu, Sindhi, Pashto)
When a critical hazard is detected, Mehfooz automatically translates actionable alerts into regional vernaculars for the downstream SMS & voice broadcast dispatchers.

In [ ]:
alert_msg = "EMERGENCY GLOF WARNING: Shishper Glacial Lake area has expanded rapidly with new drainage breaches. Move to higher ground immediately."

languages = [("ur", "Urdu"), ("sd", "Sindhi"), ("ps", "Pashto")]

for code, lang in languages:
    translated = translate_alert(alert_msg, code)
    print(f"[{lang} ({code})]: {translated}")

## 4. End-to-End Pipeline Execution
Execute full synchronous analysis run storing record in SQLite/PostgreSQL database.

In [ ]:
result = run_pipeline("passu_lake")
import pprint
pprint.pprint(result)